Libraries

In [ ]:
import re
import math
from pathlib import Path
from typing import Optional, Dict, Iterable, List, Tuple, Any
import copy
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import nbinom, lognorm, skewnorm, exponweib
from scipy.optimize import minimize
from scipy.stats import (
    poisson,
    nbinom,
    lognorm,
    gamma as gamma_dist,
    weibull_min,
    skewnorm,
    genpareto,
    burr12,
    kstest,
)

Calculating Expected losses, STD, Premiums and reserves setting

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# =========================================================
# USER INPUTS
# =========================================================
CARGO_VOLUME_INPUTS = {
    "titanium": {
        "annual_tons_mean": 286_350,
        "mean_weight_kg": 187_469.52,
    },
    "cobalt": {
        "annual_tons_mean": 228_000,
        "mean_weight_kg": 150_007.17,
    },
    "lithium": {
        "annual_tons_mean": 169_275,
        "mean_weight_kg": 112_027.95,
    },
    "rare earth": {
        "annual_tons_mean": 57_150,
        "mean_weight_kg": 37_688.51,
    },
    "gold": {
        "annual_tons_mean": 5_775,
        "mean_weight_kg": 3_810.13,
    },
    "platinum": {
        "annual_tons_mean": 3_450,
        "mean_weight_kg": 2_285.99,
    },
    "supplies": {
        "annual_tons_mean": 114_753,
        "mean_weight_kg": 75_189.57,
    },
}

# =========================================================
# MODELLING SETTINGS
# =========================================================
TONS_CV = 0.10

# Frequency model per shipment: NB2
FREQ_MU = 0.244779
FREQ_ALPHA = 1.863151

N_SIMS = 1_000
N_YEARS = 10
RANDOM_SEED = 42

# =========================================================
# PREMIUM SETTINGS
# premium = (EL + risk_loading * SD + expense_variable * EL) * (1 + profit_margin)
# =========================================================
RISK_LOADING = 0.2
EXPENSE_VARIABLE = 0.20
PROFIT_MARGIN = 0.12

# =========================================================
# STOCHASTIC 10-YEAR INTEREST RATE ASSUMPTION
# mean     = 2.8957%
# variance = 0.0122332%  (interpreted as percentage-point-squared)
# =========================================================
INTEREST_RATE_MEAN_PCT = 2.8957
INTEREST_RATE_VARIANCE_PCT2 = 0.0122332

INTEREST_RATE_MEAN = INTEREST_RATE_MEAN_PCT / 100.0
INTEREST_RATE_SD = np.sqrt(INTEREST_RATE_VARIANCE_PCT2) / 100.0

MIN_INTEREST_RATE = -0.95

# =========================================================
# BASE PRICES PER KG
# =========================================================
BASE_PRICES_PER_KG = {
    "titanium": 7.0,
    "supplies": 10.0,
    "cobalt": 52.0,
    "lithium": 82.0,
    "rare earth": 85.0,
    "gold": 54_500.0,
    "platinum": 135_600.0,
}

# =========================================================
# STOCHASTIC INFLATION ASSUMPTION
# =========================================================
INFLATION_MEAN_PCT = 2.46496
INFLATION_VARIANCE_PCT2 = 0.034085

INFLATION_MEAN = INFLATION_MEAN_PCT / 100.0
INFLATION_SD = np.sqrt(INFLATION_VARIANCE_PCT2) / 100.0

MIN_INFLATION_RATE = -0.95

# =========================================================
# CLUSTER / MINE GROWTH ASSUMPTIONS
# =========================================================
HELIONIS_BASE_PROD = 375.0
BAYESIA_BASE_PROD = 250.0
ORYN_BASE_PROD = 125.0

HELIONIS_MINES = 30
BAYESIA_MINES = 15
ORYN_MINES = 10

HELIONIS_MEAN_GROWTH = (1.25 ** (1.0 / 10.0)) - 1.0
BAYESIA_MEAN_GROWTH = (1.25 ** (1.0 / 10.0)) - 1.0
ORYN_MEAN_GROWTH = (1.15 ** (1.0 / 10.0)) - 1.0

PRODUCTION_GROWTH_SD_MULTIPLIER = 0.25

HELIONIS_GROWTH_SD = PRODUCTION_GROWTH_SD_MULTIPLIER * HELIONIS_MEAN_GROWTH
BAYESIA_GROWTH_SD = PRODUCTION_GROWTH_SD_MULTIPLIER * BAYESIA_MEAN_GROWTH
ORYN_GROWTH_SD = PRODUCTION_GROWTH_SD_MULTIPLIER * ORYN_MEAN_GROWTH

MIN_PRODUCTION_GROWTH_RATE = -0.95

# =========================================================
# SEVERITY DISTRIBUTIONS
# =========================================================
SEVERITY_MODELS = {
    "cobalt": {
        "dist": "lognormal",
        "mu": 13.061442337216045,
        "sigma": 0.8569081381822045,
    },
    "gold": {
        "dist": "logskewnorm",
        "shape": -0.7985638313815884,
        "loc": 17.68816928480843,
        "scale": 0.987915167223077,
    },
    "lithium": {
        "dist": "lognormal",
        "mu": 13.226583575398298,
        "sigma": 0.8816801497124209,
    },
    "platinum": {
        "dist": "logskewnorm",
        "shape": -0.7358336912820225,
        "loc": 16.24010114678375,
        "scale": 0.9828604406948547,
    },
    "rare earth": {
        "dist": "logskewnorm",
        "shape": 1.1745797426685858,
        "loc": 11.565385848921597,
        "scale": 1.050387375627892,
    },
    "supplies": {
        "dist": "logexpweibull",
        "a": 0.5032574573,
        "c": 2.0603365462,
        "loc_log": 10.3422251856,
        "scale_log": 1.3075827421,
    },
    "titanium": {
        "dist": "logexpweibull",
        "a": 0.5305097331,
        "c": 2.4126125838,
        "loc_log": 10.3422162683,
        "scale_log": 1.7126950865,
    },
}

# =========================================================
# EXPERIENCE TEST CASES
# Premiums charged remain fixed from the original pricing model.
# =========================================================
EXPERIENCE_CASES = {
    "Base Experience": {
        "inflation_mean_shift": 0.00,
        "inflation_sd_mult": 1.00,

        "production_mean_mult": 1.00,
        "production_sd_mult": 1.00,

        "tons_cv_mult": 1.00,

        "freq_level_mult": 1.00,
        "freq_shock_cv": 0.10,

        "alpha_level_mult": 1.00,
        "alpha_shock_cv": 0.15,

        "severity_level_mult": 1.00,
        "severity_shock_cv": 0.15,

        "expense_level_mult": 1.00,
        "expense_shock_cv": 0.05,

        "cargo_price_idio_cv": 0.05,
    },
}

# =========================================================
# FIXED INITIAL RESERVE MODEL
# We set ONE reserve at time 0 and do NOT reset it each year.
# Goal:
#   max_t P(ruin in year t) <= 1%
#
# Ruin in year t occurs if closing assets at the end of year t are below 0.
# =========================================================
TARGET_ANNUAL_RUIN_PROB = 0.01
RESERVE_SEARCH_TOLERANCE = 1_000_000.0
RESERVE_START_GUESS = 18_000_000_000.0
RESERVE_STEP_FRACTION = 0.10

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def nb2_to_scipy_params(mu, alpha):
    """
    NB2 parameterization:
        Var(N) = mu + alpha * mu^2

    SciPy mapping:
        n = 1 / alpha
        p = n / (n + mu)
    """
    n = 1.0 / alpha
    p = n / (n + mu)
    return n, p


def nb2_to_scipy_params_vector(mu, alpha):
    mu = np.maximum(mu, 1e-12)
    alpha = np.maximum(alpha, 1e-12)
    n = 1.0 / alpha
    p = n / (n + mu)
    return n, p


def lognormal_mean_one_factors(size, cv, rng):
    """
    Returns lognormal random factors with mean = 1 and a given coefficient of variation.
    """
    cv = max(float(cv), 0.0)
    if cv == 0.0:
        return np.ones(size)

    sigma2 = np.log(1.0 + cv**2)
    sigma = np.sqrt(sigma2)
    mu = -0.5 * sigma2
    return rng.lognormal(mean=mu, sigma=sigma, size=size)


def sample_base_severity(cargo_type, size, rng):
    params = SEVERITY_MODELS[cargo_type]
    dist_name = params["dist"]

    if dist_name == "lognormal":
        y = rng.normal(loc=params["mu"], scale=params["sigma"], size=size)
        return np.exp(y)

    if dist_name == "logskewnorm":
        y = stats.skewnorm.rvs(
            a=params["shape"],
            loc=params["loc"],
            scale=params["scale"],
            size=size,
            random_state=rng,
        )
        return np.exp(y)

    if dist_name == "logexpweibull":
        y = stats.exponweib.rvs(
            a=params["a"],
            c=params["c"],
            loc=params["loc_log"],
            scale=params["scale_log"],
            size=size,
            random_state=rng,
        )
        return np.exp(y)

    raise ValueError(f"Unknown severity distribution for {cargo_type}: {dist_name}")


def calculate_premium(expected_loss, std_loss, risk_loading, expense_variable, profit_margin):
    base_cost = expected_loss + risk_loading * std_loss + expense_variable * expected_loss
    return base_cost * (1.0 + profit_margin)


def tail_mean(x, q):
    threshold = np.quantile(x, q)
    tail = x[x >= threshold]
    if len(tail) == 0:
        return np.nan
    return tail.mean()


def simulate_inflation_paths(n_sims, n_years, mean_rate, sd_rate, rng, min_rate=-0.95):
    inflation_rates = rng.normal(loc=mean_rate, scale=sd_rate, size=(n_sims, n_years))
    inflation_rates = np.clip(inflation_rates, min_rate, None)
    price_multipliers = np.cumprod(1.0 + inflation_rates, axis=1)
    return inflation_rates, price_multipliers


def simulate_interest_rate_paths(n_sims, n_years, mean_rate, sd_rate, rng, min_rate=-0.95):
    rates = rng.normal(loc=mean_rate, scale=sd_rate, size=(n_sims, n_years))
    rates = np.clip(rates, min_rate, None)
    return rates


def simulate_cluster_production_paths(
    n_sims,
    n_years,
    rng,
    helionis_mines,
    bayesia_mines,
    oryn_mines,
    helionis_base_prod,
    bayesia_base_prod,
    oryn_base_prod,
    helionis_mean_growth,
    helionis_growth_sd,
    bayesia_mean_growth,
    bayesia_growth_sd,
    oryn_mean_growth,
    oryn_growth_sd,
    min_growth_rate=-0.95,
):
    helionis_growth = rng.normal(
        loc=helionis_mean_growth,
        scale=helionis_growth_sd,
        size=(n_sims, n_years),
    )
    bayesia_growth = rng.normal(
        loc=bayesia_mean_growth,
        scale=bayesia_growth_sd,
        size=(n_sims, n_years),
    )
    oryn_growth = rng.normal(
        loc=oryn_mean_growth,
        scale=oryn_growth_sd,
        size=(n_sims, n_years),
    )

    helionis_growth = np.clip(helionis_growth, min_growth_rate, None)
    bayesia_growth = np.clip(bayesia_growth, min_growth_rate, None)
    oryn_growth = np.clip(oryn_growth, min_growth_rate, None)

    helionis_prod_per_mine = helionis_base_prod * np.cumprod(1.0 + helionis_growth, axis=1)
    bayesia_prod_per_mine = bayesia_base_prod * np.cumprod(1.0 + bayesia_growth, axis=1)
    oryn_prod_per_mine = oryn_base_prod * np.cumprod(1.0 + oryn_growth, axis=1)

    total_company_prod = (
        helionis_mines * helionis_prod_per_mine
        + bayesia_mines * bayesia_prod_per_mine
        + oryn_mines * oryn_prod_per_mine
    )

    base_total_company_prod = (
        helionis_mines * helionis_base_prod
        + bayesia_mines * bayesia_base_prod
        + oryn_mines * oryn_base_prod
    )

    company_production_multiplier = total_company_prod / base_total_company_prod

    summary = {
        "hel_prod": helionis_mines * helionis_prod_per_mine,
        "bay_prod": bayesia_mines * bayesia_prod_per_mine,
        "ory_prod": oryn_mines * oryn_prod_per_mine,
        "total_prod": total_company_prod,
        "hel_growth": helionis_growth,
        "bay_growth": bayesia_growth,
        "ory_growth": oryn_growth,
    }
    return company_production_multiplier, summary


def simulate_annual_tons_for_year(base_volume_inputs, company_production_multiplier, tons_cv, rng):
    n_sims = len(company_production_multiplier)
    simulated = {}

    for cargo_type, vals in base_volume_inputs.items():
        mean_tons = vals["annual_tons_mean"] * company_production_multiplier

        if tons_cv > 0:
            sigma2 = np.log(1.0 + tons_cv**2)
            sigma = np.sqrt(sigma2)
            mu = np.log(np.maximum(mean_tons, 1e-9)) - 0.5 * sigma2
            tons = rng.lognormal(mean=mu, sigma=sigma, size=n_sims)
        else:
            tons = mean_tons.copy()

        simulated[cargo_type] = tons

    return simulated


def build_year_prices_per_kg(base_prices_per_kg, price_multiplier_vector):
    year_prices = {}
    for cargo_type, base_price in base_prices_per_kg.items():
        year_prices[cargo_type] = base_price * price_multiplier_vector
    return year_prices


def build_year_price_draws_with_idiosyncratic_shocks(
    base_prices_per_kg,
    common_price_multiplier_vector,
    cargo_price_idio_cv,
    rng,
):
    n_sims = len(common_price_multiplier_vector)
    year_prices = {}

    for cargo_type, base_price in base_prices_per_kg.items():
        idio_factor = lognormal_mean_one_factors(
            size=n_sims,
            cv=cargo_price_idio_cv,
            rng=rng,
        )
        year_prices[cargo_type] = base_price * common_price_multiplier_vector * idio_factor

    return year_prices


def compute_year_price_summary(year, year_prices_per_kg):
    rows = []
    for cargo_type, prices in year_prices_per_kg.items():
        rows.append(
            {
                "year": year,
                "cargo_type": cargo_type,
                "mean_price_per_kg": prices.mean(),
                "std_price_per_kg": prices.std(ddof=1),
                "min_price_per_kg": prices.min(),
                "max_price_per_kg": prices.max(),
            }
        )
    return pd.DataFrame(rows)


def compute_year_revenue_summary(year, simulated_tons, year_prices_per_kg):
    rows = []

    for cargo_type in simulated_tons.keys():
        revenue = simulated_tons[cargo_type] * 1000.0 * year_prices_per_kg[cargo_type]
        rows.append(
            {
                "year": year,
                "cargo_type": cargo_type,
                "mean_revenue": revenue.mean(),
                "std_revenue": revenue.std(ddof=1),
                "min_revenue": revenue.min(),
                "max_revenue": revenue.max(),
            }
        )

    revenue_df = pd.DataFrame(rows)

    ores_only = revenue_df[revenue_df["cargo_type"] != "supplies"]
    total_expected_revenue_ores_only = ores_only["mean_revenue"].sum()
    total_expected_revenue_including_supplies = revenue_df["mean_revenue"].sum()

    total_summary = pd.DataFrame(
        {
            "year": [year],
            "total_expected_revenue_ores_only": [total_expected_revenue_ores_only],
            "total_expected_revenue_including_supplies": [total_expected_revenue_including_supplies],
        }
    )

    return revenue_df, total_summary


def sample_price_scaled_severity(
    cargo_type,
    size,
    rng,
    base_prices_per_kg,
    year_prices_per_kg,
    sim_index,
):
    base_claim_sizes = sample_base_severity(cargo_type=cargo_type, size=size, rng=rng)
    base_price = base_prices_per_kg[cargo_type]
    year_prices_for_claims = year_prices_per_kg[cargo_type][sim_index]
    price_ratio = year_prices_for_claims / base_price
    return base_claim_sizes * price_ratio


def run_one_year_simulation(
    year,
    base_volume_inputs,
    base_prices_per_kg,
    company_production_multiplier,
    price_multiplier_vector,
    tons_cv,
    freq_mu,
    freq_alpha,
    risk_loading,
    expense_variable,
    profit_margin,
    rng,
):
    n_sims = len(company_production_multiplier)

    simulated_tons = simulate_annual_tons_for_year(
        base_volume_inputs=base_volume_inputs,
        company_production_multiplier=company_production_multiplier,
        tons_cv=tons_cv,
        rng=rng,
    )

    year_prices_per_kg = build_year_prices_per_kg(
        base_prices_per_kg=base_prices_per_kg,
        price_multiplier_vector=price_multiplier_vector,
    )

    portfolio_losses = np.zeros(n_sims)
    shipment_summary_rows = []

    n_ship, p_ship = nb2_to_scipy_params(freq_mu, freq_alpha)

    for cargo_type, vals in base_volume_inputs.items():
        mean_weight_kg = vals["mean_weight_kg"]
        annual_tons = simulated_tons[cargo_type]

        estimated_shipments = annual_tons * 1000.0 / mean_weight_kg
        estimated_shipments_int = np.maximum(np.round(estimated_shipments).astype(int), 0)

        shipment_summary_rows.append(
            {
                "year": year,
                "cargo_type": cargo_type,
                "mean_annual_tons": annual_tons.mean(),
                "std_annual_tons": annual_tons.std(ddof=1),
                "mean_shipments": estimated_shipments.mean(),
                "std_shipments": estimated_shipments.std(ddof=1),
            }
        )

        losses = np.zeros(n_sims)

        annual_claim_counts = stats.nbinom.rvs(
            n=n_ship,
            p=p_ship,
            size=estimated_shipments_int.sum(),
            random_state=rng,
        )

        shipment_sim_index = np.repeat(np.arange(n_sims), estimated_shipments_int)
        shipment_claim_indicator = annual_claim_counts > 0

        if shipment_claim_indicator.any():
            positive_claim_counts = annual_claim_counts[shipment_claim_indicator]
            sim_index = np.repeat(
                shipment_sim_index[shipment_claim_indicator],
                positive_claim_counts,
            )
            total_claims = int(positive_claim_counts.sum())

            claim_sizes = sample_price_scaled_severity(
                cargo_type=cargo_type,
                size=total_claims,
                rng=rng,
                base_prices_per_kg=base_prices_per_kg,
                year_prices_per_kg=year_prices_per_kg,
                sim_index=sim_index,
            )

            losses = np.bincount(
                sim_index,
                weights=claim_sizes,
                minlength=n_sims,
            ).astype(float)

        portfolio_losses += losses

    expected_loss = portfolio_losses.mean()
    std_dev = portfolio_losses.std(ddof=1)

    var_95  = np.quantile(portfolio_losses, 0.95)
    var_99  = np.quantile(portfolio_losses, 0.99)
    var_999 = np.quantile(portfolio_losses, 0.999)
    tvar_99 = tail_mean(portfolio_losses, 0.99)      # TVaR 99%

    premium = calculate_premium(
        expected_loss=expected_loss,
        std_loss=std_dev,
        risk_loading=risk_loading,
        expense_variable=expense_variable,
        profit_margin=profit_margin,
    )

    shipment_summary_df = pd.DataFrame(shipment_summary_rows).sort_values("cargo_type").reset_index(drop=True)

    price_summary_df = compute_year_price_summary(
        year=year,
        year_prices_per_kg=year_prices_per_kg,
    )

    revenue_df, total_revenue_summary_df = compute_year_revenue_summary(
        year=year,
        simulated_tons=simulated_tons,
        year_prices_per_kg=year_prices_per_kg,
    )

    summary = {
        "year": year,
        "mean_production_multiplier": company_production_multiplier.mean(),
        "std_production_multiplier": company_production_multiplier.std(ddof=1),
        "mean_price_multiplier": price_multiplier_vector.mean(),
        "std_price_multiplier": price_multiplier_vector.std(ddof=1),
        "expected_loss": expected_loss,
        "std_dev": std_dev,
        "var_95": var_95,
        "var_99": var_99,
        "var_999": var_999,
        "tvar_99": tvar_99,                          # TVaR 99%
        "premium": premium,
        "total_expected_revenue_including_supplies":
            total_revenue_summary_df.loc[0, "total_expected_revenue_including_supplies"],
        "total_expected_revenue_ores_only":
            total_revenue_summary_df.loc[0, "total_expected_revenue_ores_only"],
    }

    return shipment_summary_df, price_summary_df, revenue_df, summary


def project_10_year_premiums(
    base_volume_inputs,
    base_prices_per_kg,
    tons_cv,
    freq_mu,
    freq_alpha,
    n_sims,
    n_years,
    seed,
    risk_loading,
    expense_variable,
    profit_margin,
):
    rng = np.random.default_rng(seed)

    inflation_rates, price_multipliers = simulate_inflation_paths(
        n_sims=n_sims,
        n_years=n_years,
        mean_rate=INFLATION_MEAN,
        sd_rate=INFLATION_SD,
        rng=rng,
        min_rate=MIN_INFLATION_RATE,
    )

    company_prod_multipliers, growth_summary = simulate_cluster_production_paths(
        n_sims=n_sims,
        n_years=n_years,
        rng=rng,
        helionis_mines=HELIONIS_MINES,
        bayesia_mines=BAYESIA_MINES,
        oryn_mines=ORYN_MINES,
        helionis_base_prod=HELIONIS_BASE_PROD,
        bayesia_base_prod=BAYESIA_BASE_PROD,
        oryn_base_prod=ORYN_BASE_PROD,
        helionis_mean_growth=HELIONIS_MEAN_GROWTH,
        helionis_growth_sd=HELIONIS_GROWTH_SD,
        bayesia_mean_growth=BAYESIA_MEAN_GROWTH,
        bayesia_growth_sd=BAYESIA_GROWTH_SD,
        oryn_mean_growth=ORYN_MEAN_GROWTH,
        oryn_growth_sd=ORYN_GROWTH_SD,
        min_growth_rate=MIN_PRODUCTION_GROWTH_RATE,
    )

    all_year_summaries = []
    all_shipment_summaries = []
    all_price_summaries = []
    all_revenue_summaries = []

    for year in range(1, n_years + 1):
        year_idx = year - 1

        company_multiplier = company_prod_multipliers[:, year_idx]
        price_multiplier = price_multipliers[:, year_idx]

        shipment_summary_df, price_summary_df, revenue_df, summary = run_one_year_simulation(
            year=year,
            base_volume_inputs=base_volume_inputs,
            base_prices_per_kg=base_prices_per_kg,
            company_production_multiplier=company_multiplier,
            price_multiplier_vector=price_multiplier,
            tons_cv=tons_cv,
            freq_mu=freq_mu,
            freq_alpha=freq_alpha,
            risk_loading=risk_loading,
            expense_variable=expense_variable,
            profit_margin=profit_margin,
            rng=rng,
        )

        summary["mean_inflation_rate"] = inflation_rates[:, year_idx].mean()
        summary["std_inflation_rate"] = inflation_rates[:, year_idx].std(ddof=1)

        summary["mean_helionis_production"] = growth_summary["hel_prod"][:, year_idx].mean()
        summary["std_helionis_production"] = growth_summary["hel_prod"][:, year_idx].std(ddof=1)

        summary["mean_bayesia_production"] = growth_summary["bay_prod"][:, year_idx].mean()
        summary["std_bayesia_production"] = growth_summary["bay_prod"][:, year_idx].std(ddof=1)

        summary["mean_oryn_production"] = growth_summary["ory_prod"][:, year_idx].mean()
        summary["std_oryn_production"] = growth_summary["ory_prod"][:, year_idx].std(ddof=1)

        summary["mean_total_company_production"] = growth_summary["total_prod"][:, year_idx].mean()
        summary["std_total_company_production"] = growth_summary["total_prod"][:, year_idx].std(ddof=1)

        all_year_summaries.append(summary)
        all_shipment_summaries.append(shipment_summary_df)
        all_price_summaries.append(price_summary_df)
        all_revenue_summaries.append(revenue_df)

    premium_projection_df = pd.DataFrame(all_year_summaries)
    shipment_projection_df = pd.concat(all_shipment_summaries, ignore_index=True)
    price_projection_df = pd.concat(all_price_summaries, ignore_index=True)
    revenue_projection_df = pd.concat(all_revenue_summaries, ignore_index=True)

    premium_projection_df["premium_growth_yoy"] = premium_projection_df["premium"].pct_change()
    premium_projection_df["cumulative_premium"] = premium_projection_df["premium"].cumsum()

    return (
        premium_projection_df,
        shipment_projection_df,
        price_projection_df,
        revenue_projection_df,
        inflation_rates,
        price_multipliers,
        company_prod_multipliers,
        growth_summary,
    )


# =========================================================
# STOCHASTIC ACTUAL LOSS MODEL FOR BALANCE SHEET TEST
# =========================================================
def simulate_actual_losses_one_year_stochastic(
    base_volume_inputs,
    base_prices_per_kg,
    company_production_multiplier,
    common_price_multiplier_vector,
    tons_cv,
    mu_ship_vector,
    alpha_ship_vector,
    severity_factor_vector,
    cargo_price_idio_cv,
    rng,
):
    n_sims = len(company_production_multiplier)

    simulated_tons = simulate_annual_tons_for_year(
        base_volume_inputs=base_volume_inputs,
        company_production_multiplier=company_production_multiplier,
        tons_cv=tons_cv,
        rng=rng,
    )

    year_prices_per_kg = build_year_price_draws_with_idiosyncratic_shocks(
        base_prices_per_kg=base_prices_per_kg,
        common_price_multiplier_vector=common_price_multiplier_vector,
        cargo_price_idio_cv=cargo_price_idio_cv,
        rng=rng,
    )

    n_ship_vector, p_ship_vector = nb2_to_scipy_params_vector(
        mu=mu_ship_vector,
        alpha=alpha_ship_vector,
    )

    portfolio_losses = np.zeros(n_sims)

    for cargo_type, vals in base_volume_inputs.items():
        mean_weight_kg = vals["mean_weight_kg"]
        annual_tons = simulated_tons[cargo_type]
        estimated_shipments = annual_tons * 1000.0 / mean_weight_kg
        estimated_shipments_int = np.maximum(np.round(estimated_shipments).astype(int), 0)

        total_shipments = estimated_shipments_int.sum()
        if total_shipments == 0:
            continue

        annual_claim_counts = stats.nbinom.rvs(
            n=np.repeat(n_ship_vector, estimated_shipments_int),
            p=np.repeat(p_ship_vector, estimated_shipments_int),
            size=total_shipments,
            random_state=rng,
        )

        shipment_sim_index = np.repeat(np.arange(n_sims), estimated_shipments_int)
        shipment_claim_indicator = annual_claim_counts > 0

        if not shipment_claim_indicator.any():
            continue

        positive_claim_counts = annual_claim_counts[shipment_claim_indicator]
        sim_index = np.repeat(
            shipment_sim_index[shipment_claim_indicator],
            positive_claim_counts,
        )
        total_claims = int(positive_claim_counts.sum())

        base_claim_sizes = sample_price_scaled_severity(
            cargo_type=cargo_type,
            size=total_claims,
            rng=rng,
            base_prices_per_kg=base_prices_per_kg,
            year_prices_per_kg=year_prices_per_kg,
            sim_index=sim_index,
        )

        claim_sizes = base_claim_sizes * severity_factor_vector[sim_index]

        losses = np.bincount(
            sim_index,
            weights=claim_sizes,
            minlength=n_sims,
        ).astype(float)

        portfolio_losses += losses

    return portfolio_losses


def simulate_balance_sheet_paths_with_fixed_initial_reserve(
    case_name,
    case_settings,
    pricing_projection_df,
    n_sims,
    n_years,
    seed,
    initial_reserve,
):
    rng = np.random.default_rng(seed)

    inflation_rates, price_multipliers = simulate_inflation_paths(
        n_sims=n_sims,
        n_years=n_years,
        mean_rate=INFLATION_MEAN + case_settings["inflation_mean_shift"],
        sd_rate=INFLATION_SD * case_settings["inflation_sd_mult"],
        rng=rng,
        min_rate=MIN_INFLATION_RATE,
    )

    company_prod_multipliers, growth_summary = simulate_cluster_production_paths(
        n_sims=n_sims,
        n_years=n_years,
        rng=rng,
        helionis_mines=HELIONIS_MINES,
        bayesia_mines=BAYESIA_MINES,
        oryn_mines=ORYN_MINES,
        helionis_base_prod=HELIONIS_BASE_PROD,
        bayesia_base_prod=BAYESIA_BASE_PROD,
        oryn_base_prod=ORYN_BASE_PROD,
        helionis_mean_growth=HELIONIS_MEAN_GROWTH * case_settings["production_mean_mult"],
        helionis_growth_sd=HELIONIS_GROWTH_SD * case_settings["production_sd_mult"],
        bayesia_mean_growth=BAYESIA_MEAN_GROWTH * case_settings["production_mean_mult"],
        bayesia_growth_sd=BAYESIA_GROWTH_SD * case_settings["production_sd_mult"],
        oryn_mean_growth=ORYN_MEAN_GROWTH * case_settings["production_mean_mult"],
        oryn_growth_sd=ORYN_GROWTH_SD * case_settings["production_sd_mult"],
        min_growth_rate=MIN_PRODUCTION_GROWTH_RATE,
    )

    interest_rates = simulate_interest_rate_paths(
        n_sims=n_sims,
        n_years=n_years,
        mean_rate=INTEREST_RATE_MEAN,
        sd_rate=INTEREST_RATE_SD,
        rng=rng,
        min_rate=MIN_INTEREST_RATE,
    )

    opening_assets = np.full(n_sims, float(initial_reserve))
    ever_ruined = np.zeros(n_sims, dtype=bool)
    rows = []

    for year in range(1, n_years + 1):
        year_idx = year - 1
        pricing_row = pricing_projection_df.loc[pricing_projection_df["year"] == year].iloc[0]

        premium_collected = pricing_row["premium"]

        company_multiplier = company_prod_multipliers[:, year_idx]
        common_price_multiplier_vector = price_multipliers[:, year_idx]

        tons_cv = TONS_CV * case_settings["tons_cv_mult"]

        freq_shock = lognormal_mean_one_factors(
            size=n_sims,
            cv=case_settings["freq_shock_cv"],
            rng=rng,
        )
        mu_ship_vector = FREQ_MU * case_settings["freq_level_mult"] * freq_shock

        alpha_shock = lognormal_mean_one_factors(
            size=n_sims,
            cv=case_settings["alpha_shock_cv"],
            rng=rng,
        )
        alpha_ship_vector = FREQ_ALPHA * case_settings["alpha_level_mult"] * alpha_shock

        severity_shock = lognormal_mean_one_factors(
            size=n_sims,
            cv=case_settings["severity_shock_cv"],
            rng=rng,
        )
        severity_factor_vector = case_settings["severity_level_mult"] * severity_shock

        expense_shock = lognormal_mean_one_factors(
            size=n_sims,
            cv=case_settings["expense_shock_cv"],
            rng=rng,
        )
        expense_ratio_vector = EXPENSE_VARIABLE * case_settings["expense_level_mult"] * expense_shock

        actual_losses = simulate_actual_losses_one_year_stochastic(
            base_volume_inputs=CARGO_VOLUME_INPUTS,
            base_prices_per_kg=BASE_PRICES_PER_KG,
            company_production_multiplier=company_multiplier,
            common_price_multiplier_vector=common_price_multiplier_vector,
            tons_cv=tons_cv,
            mu_ship_vector=mu_ship_vector,
            alpha_ship_vector=alpha_ship_vector,
            severity_factor_vector=severity_factor_vector,
            cargo_price_idio_cv=case_settings["cargo_price_idio_cv"],
            rng=rng,
        )

        underwriting_expense_vector = premium_collected * expense_ratio_vector

        underwriting_profit_vector = (
            premium_collected
            - actual_losses
            - underwriting_expense_vector
        )

        investment_income_vector = opening_assets * interest_rates[:, year_idx]
        total_profit_vector = underwriting_profit_vector + investment_income_vector
        closing_assets = opening_assets + total_profit_vector

        ruined_this_year = closing_assets < 0.0
        ever_ruined = ever_ruined | ruined_this_year

        shortfall_vector = np.maximum(-closing_assets, 0.0)

        combined_ratio_vector = (
            actual_losses
            + underwriting_expense_vector
        ) / premium_collected

        row = {
            "case": case_name,
            "year": year,
            "initial_reserve": float(initial_reserve),
            "premium_collected": premium_collected,
            "mean_opening_assets": opening_assets.mean(),
            "mean_interest_rate": interest_rates[:, year_idx].mean(),
            "sd_interest_rate": interest_rates[:, year_idx].std(ddof=1),
            "mean_investment_income": investment_income_vector.mean(),
            "mean_underwriting_expense": underwriting_expense_vector.mean(),
            "mean_actual_loss": actual_losses.mean(),
            "sd_actual_loss": actual_losses.std(ddof=1),
            "var95_loss": np.quantile(actual_losses, 0.95),
            "var99_loss": np.quantile(actual_losses, 0.99),
            "var999_loss": np.quantile(actual_losses, 0.999),
            "cte95_loss": tail_mean(actual_losses, 0.95),
            "cte99_loss": tail_mean(actual_losses, 0.99),
            "mean_underwriting_profit": underwriting_profit_vector.mean(),
            "mean_total_profit": total_profit_vector.mean(),
            "p01_total_profit": np.quantile(total_profit_vector, 0.01),
            "p05_total_profit": np.quantile(total_profit_vector, 0.05),
            "mean_closing_assets": closing_assets.mean(),
            "p01_closing_assets": np.quantile(closing_assets, 0.01),
            "p05_closing_assets": np.quantile(closing_assets, 0.05),
            "prob_underwriting_loss": np.mean(underwriting_profit_vector < 0.0),
            "prob_total_loss": np.mean(total_profit_vector < 0.0),
            "prob_ruin_in_year": np.mean(ruined_this_year),
            "prob_ruin_to_date": np.mean(ever_ruined),
            "mean_shortfall": shortfall_vector.mean(),
            "conditional_mean_shortfall": (
                shortfall_vector[shortfall_vector > 0.0].mean()
                if np.any(shortfall_vector > 0.0) else 0.0
            ),
            "mean_combined_ratio": combined_ratio_vector.mean(),
            "p95_combined_ratio": np.quantile(combined_ratio_vector, 0.95),
            "mean_inflation_rate": inflation_rates[:, year_idx].mean(),
            "sd_inflation_rate": inflation_rates[:, year_idx].std(ddof=1),
            "mean_total_company_production": growth_summary["total_prod"][:, year_idx].mean(),
            "sd_total_company_production": growth_summary["total_prod"][:, year_idx].std(ddof=1),
        }

        rows.append(row)
        opening_assets = closing_assets.copy()

    result_df = pd.DataFrame(rows)
    worst_annual_ruin_prob = float(result_df["prob_ruin_in_year"].max())

    return result_df, worst_annual_ruin_prob


def find_required_initial_reserve(
    case_name,
    case_settings,
    pricing_projection_df,
    n_sims,
    n_years,
    seed,
    target_annual_ruin_prob=0.01,
    tolerance=1_000_000.0,
    start_guess=20_000_000_000.0,
    step_fraction=0.10,
    max_iter=60,
):
    """
    Find the LOWEST initial reserve such that:

        max over years of P(ruin in year t) <= target_annual_ruin_prob

    The search starts around start_guess to reduce runtime.
    """
    cache = {}

    def evaluate(reserve):
        reserve = float(max(0.0, reserve))
        key = round(reserve, 2)

        if key not in cache:
            result_df, worst_annual_ruin_prob = simulate_balance_sheet_paths_with_fixed_initial_reserve(
                case_name=case_name,
                case_settings=case_settings,
                pricing_projection_df=pricing_projection_df,
                n_sims=n_sims,
                n_years=n_years,
                seed=seed,
                initial_reserve=reserve,
            )

            final_ruin_to_date = float(result_df["prob_ruin_to_date"].iloc[-1])

            cache[key] = {
                "reserve": reserve,
                "result_df": result_df,
                "worst_annual_ruin_prob": float(worst_annual_ruin_prob),
                "final_ruin_to_date": final_ruin_to_date,
            }

        return cache[key]

    guess = float(start_guess)
    guess_eval = evaluate(guess)

    # Step 1: bracket the solution around the starting guess
    if guess_eval["worst_annual_ruin_prob"] <= target_annual_ruin_prob:
        high = guess
        low = max(0.0, guess * (1.0 - step_fraction))
        low_eval = evaluate(low)

        while low > 0.0 and low_eval["worst_annual_ruin_prob"] <= target_annual_ruin_prob:
            high = low
            low = max(0.0, low * (1.0 - step_fraction))
            low_eval = evaluate(low)
    else:
        low = guess
        high = guess * (1.0 + step_fraction)
        high_eval = evaluate(high)

        while high_eval["worst_annual_ruin_prob"] > target_annual_ruin_prob:
            low = high
            high = high * (1.0 + step_fraction)
            high_eval = evaluate(high)

            if high > 1e15:
                raise RuntimeError("Could not find an upper bound for the initial reserve.")

    # Step 2: binary search inside the bracket
    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        mid_eval = evaluate(mid)

        if mid_eval["worst_annual_ruin_prob"] <= target_annual_ruin_prob:
            high = mid
        else:
            low = mid

        if (high - low) <= tolerance:
            break

    best_eval = evaluate(high)

    return (
        best_eval["reserve"],
        best_eval["worst_annual_ruin_prob"],
        best_eval["final_ruin_to_date"],
        best_eval["result_df"],
    )


# =========================================================
# RUN BASE PRICING MODEL
# =========================================================
(
    premium_projection_df,
    shipment_projection_df,
    price_projection_df,
    revenue_projection_df,
    inflation_rates,
    price_multipliers,
    company_prod_multipliers,
    growth_summary,
) = project_10_year_premiums(
    base_volume_inputs=CARGO_VOLUME_INPUTS,
    base_prices_per_kg=BASE_PRICES_PER_KG,
    tons_cv=TONS_CV,
    freq_mu=FREQ_MU,
    freq_alpha=FREQ_ALPHA,
    n_sims=N_SIMS,
    n_years=N_YEARS,
    seed=RANDOM_SEED,
    risk_loading=RISK_LOADING,
    expense_variable=EXPENSE_VARIABLE,
    profit_margin=PROFIT_MARGIN,
)

# =========================================================
# FIND REQUIRED FIXED INITIAL RESERVE - BASE CASE ONLY
# =========================================================
(
    base_required_reserve,
    base_worst_annual_ruin_prob,
    base_final_ruin_to_date,
    base_result_at_own_reserve,
) = find_required_initial_reserve(
    case_name="Base Experience",
    case_settings=EXPERIENCE_CASES["Base Experience"],
    pricing_projection_df=premium_projection_df,
    n_sims=N_SIMS,
    n_years=N_YEARS,
    seed=RANDOM_SEED + 100,
    target_annual_ruin_prob=TARGET_ANNUAL_RUIN_PROB,
    tolerance=RESERVE_SEARCH_TOLERANCE,
    start_guess=RESERVE_START_GUESS,
    step_fraction=RESERVE_STEP_FRACTION,
)

# =========================================================
# FINAL CONDENSED OUTPUT TABLE
# =========================================================
final_output_table = (
    premium_projection_df[
        [
            "year",
            "expected_loss",
            "std_dev",
            "var_95",
            "var_99",
            "var_999",
            "tvar_99",              # TVaR 99% from pricing simulation
            "cumulative_premium",
        ]
    ]
    .merge(
        base_result_at_own_reserve[
            [
                "year",
                "initial_reserve",
                "prob_ruin_in_year",
                "prob_ruin_to_date",
                "mean_closing_assets",
                "mean_investment_income",
                "mean_total_profit",
            ]
        ],
        on="year",
        how="left",
    )
    .rename(
        columns={
            "std_dev": "std_dev_loss",
            "tvar_99": "tvar_99_loss",
            "cumulative_premium": "cumulative_premiums_collected",
            "initial_reserve": "required_initial_reserve",
            "prob_ruin_in_year": "probability_of_ruin_in_year",
            "prob_ruin_to_date": "probability_of_ruin_to_date",
            "mean_closing_assets": "expected_end_of_year_reserve",
            "mean_investment_income": "investment_income_from_reserves",
            "mean_total_profit": "profit_of_policy_per_year",
        }
    )
)

final_output_table["cumulative_expected_profit"] = final_output_table["profit_of_policy_per_year"].cumsum()
total_profit_10_years = float(final_output_table["profit_of_policy_per_year"].sum())

pd.options.display.float_format = "{:,.4f}".format

print("\n=== BASE CASE REQUIRED INITIAL RESERVE ===")
print(f"Required initial reserve = {base_required_reserve:,.2f}")
print(f"Worst annual ruin probability at this reserve = {base_worst_annual_ruin_prob:.4%}")
print(f"Final ruin-to-date probability over 10 years = {base_final_ruin_to_date:.4%}")

print("\n=== CONDENSED BASE CASE RESULTS ===")
print(
    final_output_table[
        [
            "year",
            "expected_loss",
            "std_dev_loss",
            "var_95",
            "var_99",
            "tvar_99_loss",                 # TVaR 99%
            "var_999",
            "cumulative_premiums_collected",
            "required_initial_reserve",
            "probability_of_ruin_in_year",
            "probability_of_ruin_to_date",
            "expected_end_of_year_reserve",
            "investment_income_from_reserves",
            "profit_of_policy_per_year",
            "cumulative_expected_profit",
        ]
    ]
)

print("\n=== TOTAL EXPECTED PROFIT AT END OF 10 YEARS ===")
print(f"{total_profit_10_years:,.2f}")

# =========================================================
# PLOTS
# =========================================================
plt.figure(figsize=(10, 6))
plt.plot(
    final_output_table["year"],
    final_output_table["cumulative_expected_profit"],
    marker="o",
)
plt.title("Cumulative Expected Profit by Year")
plt.xlabel("Year")
plt.ylabel("Cumulative expected profit")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(
    final_output_table["year"],
    final_output_table["probability_of_ruin_in_year"],
    marker="o",
    label="Annual ruin probability",
)
plt.axhline(
    TARGET_ANNUAL_RUIN_PROB,
    linestyle="--",
    label="1% target",
)
plt.title("Probability of Ruin in Each Year")
plt.xlabel("Year")
plt.ylabel("Probability of ruin")
plt.legend()
plt.tight_layout()
plt.show()

# TVaR 99% vs VaR 99% vs VaR 99.9% over time
plt.figure(figsize=(10, 6))
plt.plot(
    final_output_table["year"],
    final_output_table["var_99"],
    marker="o",
    label="VaR 99%",
)
plt.plot(
    final_output_table["year"],
    final_output_table["tvar_99_loss"],
    marker="s",
    label="TVaR 99%",
    linestyle="--",
)
plt.plot(
    final_output_table["year"],
    final_output_table["var_999"],
    marker="^",
    label="VaR 99.9%",
    linestyle=":",
)
plt.title("Tail Risk Metrics by Year (Pricing Simulation)")
plt.xlabel("Year")
plt.ylabel("Loss ($)")
plt.legend()
plt.tight_layout()
plt.show()